# Latent Probe Diagnostic

Tests how much task-relevant information lives in the world-model latents.
For every step in every eval episode we extract two latent trajectories:

1. **Posterior** — drive encoder + RSSM with the real obs stream
   (`obs_step` at every step). Latents reflect what the encoder + posterior
   net can extract directly from observations.
2. **Prior** — burn the posterior in for ``burn_in`` steps, then continue
   open-loop with the prior (`s = prior_mean`, no obs). Latents now reflect
   what the dynamics alone can predict starting from the burned-in state.

For each variant we train two probes — a **linear** one and a small **MLP** —
from latent features to three ground-truth targets recorded with each episode:

* `object_xy` — pushed object position (2-D)
* `ee_xy` — end-effector xy (2-D; the planar component of `ee_pos`)
* `joint_qpos` — one probe per joint angle

The latent inputs we sweep over are ``h_t`` alone, ``s_t`` alone, and the
concatenation ``[h_t, s_t]`` (i.e. the same `feat` the decoder consumes).
Higher R² on the test split = more information about that quantity is
linearly (or near-linearly) decodable from the latent.

We hold out a fraction of episodes for testing so the train/test split is
across trajectories rather than across timesteps within a trajectory.

In [ ]:
# Parameters — papermill injects overrides after this cell.
checkpoint_path = "checkpoints/default/final.safetensors"
data_path       = "data/eval"
data_format     = "rerun"   # "hdf5" or "rerun"
num_episodes    = 40
burn_in         = 5
horizon         = 15        # open-loop prior horizon after burn-in
test_frac       = 0.25      # fraction of episodes used for probe test set
mlp_hidden      = (64, 64)
mlp_epochs      = 200
mlp_lr          = 1e-3
mlp_batch       = 512
ridge_alpha     = 1e-3      # L2 reg for the linear (ridge) probe
seed            = 0

## Setup

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import mlx.core as mx
import mlx.nn as mlxnn
import mlx.optimizers as mlxopt
import numpy as np

from chuck_dreamer.dreamer import build_model, rollout
from chuck_dreamer.dreamer.mlx_model import load_config_from_checkpoint
from chuck_dreamer.sim import PushingEnv
from chuck_dreamer.training.episode_loader import iter_episodes
from chuck_dreamer.training.episode_processor import processor_for

rng = np.random.default_rng(seed)
mx.random.seed(seed)

print(f"checkpoint:   {checkpoint_path}")
print(f"data:         {data_path} ({data_format})")
print(f"num_episodes: {num_episodes}")
print(f"burn_in:      {burn_in}, horizon: {horizon}")
print(f"test_frac:    {test_frac}")

## Load model and evaluation episodes

We reuse the same loader + processor the replay buffer uses so the encoder
sees identical inputs to training. The raw episode (pre-processor) is also
kept around: the processor's `obs` is the modal observation (image, state,
or dict), but our ground truths — object xy / ee pos / joint angles — live
in the raw episode and may be discarded by the modal processor.

The buffer convention is `obs[t+1]` paired with `action[t]` produces RSSM
state at index `t`. So latent step `t` lines up with the obs frame *after*
action `t`, which is exactly what `joint_qpos[t]`, `ee_pos[t]`, `object_xy[t]`
record in the raw episode.

In [ ]:
config = load_config_from_checkpoint(checkpoint_path)
if config is None:
  raise RuntimeError(
    f"checkpoint {checkpoint_path!r} has no embedded config metadata; "
    "re-train or re-save with a current Trainer to embed the config."
  )

env        = PushingEnv(config)
obs_shape  = env.model_obs_shape
action_dim = int(env.action_space.shape[0])
obs_mode   = env.obs_mode

model = build_model(config, obs_shape=obs_shape, action_dim=action_dim)
model.training = False
model.load(checkpoint_path)

stoch_dim = model.rssm.stoch_dim
deter_dim = model.rssm.deter_dim
print(f"Loaded {checkpoint_path}")
print(f"obs_mode={obs_mode}  act_mode={env.act_mode}")
print(f"RSSM dims: stoch={stoch_dim}, deter={deter_dim}")

In [ ]:
# We need both the processed episode (for the encoder's obs format) and the
# raw episode (for the ground-truth columns the modal processor drops).
processor = processor_for(config)
min_len   = burn_in + horizon

raw_episodes: list[dict] = []
episodes:     list[dict] = []  # processed
for raw in iter_episodes(data_path, format=data_format):
  ep = processor(raw)
  if ep["action"].shape[0] < min_len:
    continue
  raw_episodes.append(raw)
  episodes.append(ep)
  if len(episodes) >= num_episodes:
    break

n_joints = raw_episodes[0]["joint_qpos"].shape[-1]
print(f"Using {len(episodes)} episodes (>= {min_len} steps each); n_joints={n_joints}")

## Latent extraction

For each episode we run the world model twice through :func:`rollout`:

  * a full **posterior** rollout — the latent is refined with the real obs
    embedding at every step.
  * a **burn-in posterior, then prior** rollout — open-loop after burn-in,
    mimicking what the actor sees during imagination-based training.

Both use ``sample=False`` so the stochastic latent is the distribution
mean (deterministic). :meth:`Trajectory.stack_h` and
:meth:`Trajectory.stack_s` give us ``(T, deter_dim)`` /
``(T, stoch_dim)`` arrays without an explicit per-step stacking loop.

RSSM timing convention: ``state_t = obs_step(state_{t-1}, action_{t-1},
embed_t)`` where ``embed_t`` is the embedding of the observation AFTER
``action_{t-1}``. The replay buffer stores ``obs[t]`` BEFORE ``action[t]``,
so we feed ``embeds[:, 1:]``.

In [ ]:
def _batch_obs(obs):
  if isinstance(obs, dict):
    return {k: mx.array(v[None]) for k, v in obs.items()}
  return mx.array(obs[None])


def extract_latents(model, obs, actions, burn_in):
  """Run one episode through the RSSM in two modes.

  Returns dict with arrays of shape ``(T, deter_dim)`` / ``(T, stoch_dim)``:
    h_post,  s_post:  posterior rollout for every step (teacher-forced).
    h_prior, s_prior: posterior for steps [0, burn_in), prior for
                      [burn_in, T) — what the actor sees during imagination.
  All ``s`` vectors are the distribution mean (no sampling).
  """
  T = actions.shape[0]
  obs_b  = _batch_obs(obs)
  act_b  = mx.array(actions[None])                # (1, T, A)
  embeds = model.encode(obs_b)[:, 1:T + 1]        # embed of obs AFTER action_t

  post = rollout(
    model, init_state=model.initial_state(1), horizon=T,
    mode="posterior", actions=act_b, embeds=embeds, sample=False,
  )
  prior_split = rollout(
    model, init_state=model.initial_state(1), horizon=T,
    mode="prior", burn_in=burn_in,
    actions=act_b, embeds=embeds, sample=False,
  )

  h_post  = np.asarray(post.stack_h()[0])         # (T, deter_dim)
  s_post  = np.asarray(post.stack_s()[0])         # (T, stoch_dim)
  h_prior = np.asarray(prior_split.stack_h()[0])
  s_prior = np.asarray(prior_split.stack_s()[0])

  return {
    "h_post":  h_post,
    "s_post":  s_post,
    "h_prior": h_prior,
    "s_prior": s_prior,
  }

In [ ]:
# Roll every episode, collecting latents + aligned ground-truth targets.
# Targets are post-action quantities recorded in the raw episode: index t in
# the raw arrays corresponds to RSSM step t (state after action_t).
per_episode: list[dict] = []
for ep_idx, (raw, ep) in enumerate(zip(raw_episodes, episodes)):
  T = ep["action"].shape[0]
  lat = extract_latents(model, ep["obs"], ep["action"], burn_in)

  object_xy = np.asarray(raw["object_xy"], dtype=np.float32)[:T]    # (T, 2)
  ee_pos    = np.asarray(raw["ee_pos"],    dtype=np.float32)[:T]    # (T, 3)
  joints    = np.asarray(raw["joint_qpos"], dtype=np.float32)[:T]   # (T, n_joints)
  ee_xy     = ee_pos[:, :2]

  per_episode.append({
    "T":         T,
    **lat,
    "object_xy": object_xy,
    "ee_xy":     ee_xy,
    "joints":    joints,
  })
print(f"Extracted latents and targets for {len(per_episode)} episodes")

## Pool & split

We split **at the episode level**: a held-out set of episodes never
contributes any timestep to the training set. Within an episode timesteps
are highly correlated, so a within-episode split would let probes cheat
by interpolating between consecutive frames.

We further track *where in the rollout* each timestep came from
(`t_idx ∈ [0, T)`) and whether it is in the open-loop region
(`t_idx >= burn_in`). This lets us slice probe accuracy by horizon at
evaluation time — the metric of interest for the prior is how well the
*open-loop* latents preserve task quantities, not the burn-in window where
prior and posterior are identical by construction.

In [ ]:
ep_indices = np.arange(len(per_episode))
rng.shuffle(ep_indices)
n_test = max(1, int(round(test_frac * len(ep_indices))))
test_eps  = set(ep_indices[:n_test].tolist())
train_eps = set(ep_indices[n_test:].tolist())
print(f"Train episodes: {len(train_eps)}  |  Test episodes: {len(test_eps)}")


def _pool(eps, key, source):
  """Concatenate latent features across episode indices ``eps``.

  ``key`` ∈ {"h", "s", "hs"} picks the latent variant; ``source`` ∈
  {"post", "prior"} picks the rollout to draw from.
  Returns (X, t_idx) where t_idx records the within-episode step.
  """
  xs, ts = [], []
  for i in eps:
    rec = per_episode[i]
    h = rec[f"h_{source}"]
    s = rec[f"s_{source}"]
    if key == "h":
      xs.append(h)
    elif key == "s":
      xs.append(s)
    else:  # "hs"
      xs.append(np.concatenate([h, s], axis=-1))
    ts.append(np.arange(rec["T"], dtype=np.int32))
  return np.concatenate(xs, axis=0), np.concatenate(ts, axis=0)


def _pool_target(eps, target_key, col=None):
  """Concatenate a target across episodes. ``col`` picks a single column
  (used for per-joint probes), otherwise returns the full (N, D) array."""
  ys = []
  for i in eps:
    y = per_episode[i][target_key]
    if col is not None:
      y = y[:, col:col + 1]
    ys.append(y)
  return np.concatenate(ys, axis=0)

## Probe implementations

Both probes standardize inputs and targets (z-score) using train statistics,
fit, then report R² and RMSE on the raw (un-standardized) targets so numbers
are comparable across targets with different scales.

* **Linear probe** — closed-form ridge regression. With M ≪ N this is
  exact and avoids any optimization noise.
* **MLP probe** — small MLX MLP, plain Adam + MSE, full-data minibatches.
  Uses the same input standardization. Two hidden layers of size 64 by
  default — large enough to pick up curvature but small enough that it
  cannot serve as a parallel decoder.

`r2` is the standard coefficient of determination on the *aggregated* target
(matches `sklearn.metrics.r2_score(... multioutput='variance_weighted')` for
vector targets — but since we run per-output probes here, it reduces to the
scalar 1 - SSE/SST formulation).

In [ ]:
def _standardize(x_train, y_train):
  x_mean = x_train.mean(0, keepdims=True)
  x_std  = x_train.std(0, keepdims=True) + 1e-6
  y_mean = y_train.mean(0, keepdims=True)
  y_std  = y_train.std(0, keepdims=True) + 1e-6
  return x_mean, x_std, y_mean, y_std


def r2_score(y_true, y_pred):
  ss_res = float(((y_true - y_pred) ** 2).sum())
  ss_tot = float(((y_true - y_true.mean(0, keepdims=True)) ** 2).sum())
  return 1.0 - ss_res / max(ss_tot, 1e-12)


def rmse(y_true, y_pred):
  return float(np.sqrt(((y_true - y_pred) ** 2).mean()))


def fit_linear_probe(X_train, y_train, X_test, y_test, alpha):
  """Closed-form ridge in standardized space. Returns (y_pred_train,
  y_pred_test) in the *original* target space."""
  x_mean, x_std, y_mean, y_std = _standardize(X_train, y_train)
  Xs = (X_train - x_mean) / x_std
  Ys = (y_train - y_mean) / y_std

  # add bias column
  Xs_b = np.concatenate([Xs, np.ones((Xs.shape[0], 1), dtype=Xs.dtype)], axis=1)
  d = Xs_b.shape[1]
  A = Xs_b.T @ Xs_b + alpha * np.eye(d, dtype=Xs_b.dtype)
  A[-1, -1] = A[-1, -1] - alpha  # don't regularize bias
  B = Xs_b.T @ Ys
  W = np.linalg.solve(A, B)

  def _apply(X):
    Xs = (X - x_mean) / x_std
    Xs_b = np.concatenate([Xs, np.ones((Xs.shape[0], 1), dtype=Xs.dtype)], axis=1)
    Y_hat = Xs_b @ W
    return Y_hat * y_std + y_mean

  return _apply(X_train), _apply(X_test), _apply


class _ProbeMLP(mlxnn.Module):
  def __init__(self, in_dim, hidden, out_dim):
    super().__init__()
    layers: list = []
    prev = in_dim
    for h in hidden:
      layers.append(mlxnn.Linear(prev, h))
      layers.append(mlxnn.ReLU())
      prev = h
    layers.append(mlxnn.Linear(prev, out_dim))
    self.net = mlxnn.Sequential(*layers)

  def __call__(self, x):
    return self.net(x)


def fit_mlp_probe(X_train, y_train, X_test, y_test, *, hidden, epochs, lr, batch):
  x_mean, x_std, y_mean, y_std = _standardize(X_train, y_train)
  Xs = (X_train - x_mean) / x_std
  Ys = (y_train - y_mean) / y_std
  Xt = (X_test  - x_mean) / x_std

  in_dim  = Xs.shape[1]
  out_dim = Ys.shape[1]
  net = _ProbeMLP(in_dim, hidden, out_dim)
  opt = mlxopt.Adam(learning_rate=lr)

  def loss_fn(model, x, y):
    pred = model(x)
    return ((pred - y) ** 2).mean()

  loss_and_grad = mlxnn.value_and_grad(net, loss_fn)

  X_mx = mx.array(Xs.astype(np.float32))
  Y_mx = mx.array(Ys.astype(np.float32))
  n    = X_mx.shape[0]

  for _ in range(epochs):
    perm = np.random.permutation(n)
    for start in range(0, n, batch):
      idx  = mx.array(perm[start:start + batch])
      xb   = X_mx[idx]
      yb   = Y_mx[idx]
      loss, grads = loss_and_grad(net, xb, yb)
      opt.update(net, grads)
      mx.eval(net.parameters(), opt.state)

  def _apply(X):
    Xs = (X - x_mean) / x_std
    pred = np.asarray(net(mx.array(Xs.astype(np.float32))))
    return pred * y_std + y_mean

  return _apply(X_train), _apply(X_test), _apply

## Sweep

For every combination of (source ∈ {posterior, prior}) × (latent ∈ {h, s, hs}) ×
(probe ∈ {linear, mlp}) × (target ∈ {object_xy, ee_xy, joint 0..n_joints-1})
we fit on the training-episode pool and report R²/RMSE on the held-out
episode pool. We additionally restrict the test set to the open-loop region
(`t_idx >= burn_in`) for the prior source — that's the regime where prior
and posterior actually diverge.

This is the costly cell; the MLP probes dominate runtime. For ``object_xy``
and ``ee_xy`` the target is 2-D (predicted jointly by one probe); for joints
we fit one probe per joint so each one is a scalar regression.

In [ ]:
LATENT_KEYS  = ("h", "s", "hs")
SOURCES      = ("post", "prior")
PROBES       = ("linear", "mlp")


def _open_loop_mask(t_idx, source):
  """Mask selecting which timesteps count toward the eval metric."""
  if source == "post":
    return np.ones_like(t_idx, dtype=bool)
  return t_idx >= burn_in


def _run_one(target_name, target_key, col, eps_tr, eps_te):
  rows = []
  y_tr = _pool_target(eps_tr, target_key, col=col)
  y_te = _pool_target(eps_te, target_key, col=col)
  for source in SOURCES:
    for key in LATENT_KEYS:
      X_tr, t_tr = _pool(eps_tr, key, source)
      X_te, t_te = _pool(eps_te, key, source)
      mask_tr = _open_loop_mask(t_tr, source)
      mask_te = _open_loop_mask(t_te, source)

      # Train on the full set so probes always see burn-in *and* open-loop
      # timesteps. The eval mask narrows scoring to the regime of interest.
      _, lin_pred_te, _ = fit_linear_probe(X_tr, y_tr, X_te, y_te, alpha=ridge_alpha)
      _, mlp_pred_te, _ = fit_mlp_probe(X_tr, y_tr, X_te, y_te,
                                        hidden=mlp_hidden, epochs=mlp_epochs,
                                        lr=mlp_lr, batch=mlp_batch)
      for probe_name, pred in (("linear", lin_pred_te), ("mlp", mlp_pred_te)):
        rows.append({
          "target": target_name,
          "source": source,
          "latent": key,
          "probe":  probe_name,
          "r2":     r2_score(y_te[mask_te], pred[mask_te]),
          "rmse":   rmse(y_te[mask_te], pred[mask_te]),
          "n_eval": int(mask_te.sum()),
        })
  return rows


results: list[dict] = []
eps_tr = sorted(train_eps)
eps_te = sorted(test_eps)

print("Probing object_xy ...")
results += _run_one("object_xy", "object_xy", None, eps_tr, eps_te)
print("Probing ee_xy ...")
results += _run_one("ee_xy", "ee_xy", None, eps_tr, eps_te)
print(f"Probing {n_joints} joints ...")
for j in range(n_joints):
  results += _run_one(f"q_{j}", "joints", j, eps_tr, eps_te)

print(f"Total probe results: {len(results)}")

## Headline view — task-relevant targets

`object_xy` is the hardest information to recover (depends on contact
dynamics, not directly observable from proprio); `ee_xy` is mid-difficulty
(close to a fixed function of joint angles); joints are the easiest because
the action stream conditions them strongly. The gap between posterior and
open-loop prior on `object_xy` is the metric that matters most for
imagination-based actor training.

In [ ]:
def _by(rows, **filters):
  out = []
  for r in rows:
    if all(r[k] == v for k, v in filters.items()):
      out.append(r)
  return out


def _plot_grouped_r2(rows, title):
  """Grouped bar chart: x = latent ∈ {h, s, hs}, hue = source × probe."""
  variants = [("post", "linear"), ("post", "mlp"),
              ("prior", "linear"), ("prior", "mlp")]
  colors   = {"post": "C0", "prior": "C3"}
  hatches  = {"linear": "", "mlp": "//"}
  width    = 0.18
  xs       = np.arange(len(LATENT_KEYS))

  fig, ax = plt.subplots(figsize=(8, 4))
  for k, (src, probe) in enumerate(variants):
    vals = []
    for key in LATENT_KEYS:
      hit = _by(rows, source=src, latent=key, probe=probe)
      vals.append(hit[0]["r2"] if hit else np.nan)
    offset = (k - (len(variants) - 1) / 2) * width
    ax.bar(xs + offset, vals, width,
           color=colors[src], edgecolor="black", hatch=hatches[probe],
           label=f"{src} / {probe}")
  ax.set_xticks(xs); ax.set_xticklabels(LATENT_KEYS)
  ax.set_xlabel("latent input")
  ax.set_ylabel("R² (test)")
  ax.set_ylim(min(-0.05, ax.get_ylim()[0]), 1.05)
  ax.axhline(0, color="k", linewidth=0.5, alpha=0.4)
  ax.set_title(title)
  ax.legend(ncol=2, fontsize=8); ax.grid(alpha=0.3, axis="y")
  plt.tight_layout(); plt.show()


for target_name in ("object_xy", "ee_xy"):
  hit = _by(results, target=target_name)
  _plot_grouped_r2(hit, f"{target_name}: probe R² by latent / source")

## Joint values — per-joint probe scores

One row per joint. The point is to see whether the latent's joint-angle
information degrades uniformly under prior rollouts or whether some joints
(e.g. those driven hardest by the current action) are easier than others.

In [ ]:
def _joint_matrix(rows, probe, metric="r2"):
  """Build a (n_joints, 6) matrix indexed by (source × latent)."""
  cols = [(s, k) for s in SOURCES for k in LATENT_KEYS]
  M = np.full((n_joints, len(cols)), np.nan)
  for j in range(n_joints):
    target = f"q_{j}"
    for c, (src, key) in enumerate(cols):
      hit = _by(rows, target=target, source=src, latent=key, probe=probe)
      if hit:
        M[j, c] = hit[0][metric]
  col_labels = [f"{s}/{k}" for s, k in cols]
  return M, col_labels


fig, axes = plt.subplots(1, 2, figsize=(11, max(2.5, 0.35 * n_joints + 1.5)))
for ax, probe in zip(axes, ("linear", "mlp")):
  M, col_labels = _joint_matrix(results, probe, metric="r2")
  im = ax.imshow(M, vmin=-0.1, vmax=1.0, aspect="auto", cmap="viridis")
  ax.set_xticks(np.arange(len(col_labels)))
  ax.set_xticklabels(col_labels, rotation=45, ha="right")
  ax.set_yticks(np.arange(n_joints))
  ax.set_yticklabels([f"q_{j}" for j in range(n_joints)])
  ax.set_title(f"per-joint R² — {probe} probe")
  for j in range(n_joints):
    for c in range(len(col_labels)):
      ax.text(c, j, f"{M[j, c]:.2f}", ha="center", va="center",
              color="white" if M[j, c] < 0.5 else "black", fontsize=7)
fig.colorbar(im, ax=axes, shrink=0.8, label="R²")
plt.show()

## Prior rollout: probe R² vs horizon

Take the best prior probe per target (the MLP on `[h, s]`) and re-evaluate
its R² at each open-loop step `k = t_idx - burn_in`. Going from burn-in to
fully imagined, how fast does decodable information decay?

In [ ]:
def _r2_by_step(target_name, target_key, col, eps_tr, eps_te):
  """Fit the MLP probe once, then score R² stratified by step.

  Returns an array of shape (T_max,) with R² at each step (NaN if no data).
  Uses the prior latents (h+s concatenated).
  """
  X_tr, _    = _pool(eps_tr, "hs", "prior")
  X_te, t_te = _pool(eps_te, "hs", "prior")
  y_tr = _pool_target(eps_tr, target_key, col=col)
  y_te = _pool_target(eps_te, target_key, col=col)
  _, pred_te, _ = fit_mlp_probe(X_tr, y_tr, X_te, y_te,
                                hidden=mlp_hidden, epochs=mlp_epochs,
                                lr=mlp_lr, batch=mlp_batch)
  T_max = int(t_te.max()) + 1
  out = np.full(T_max, np.nan)
  for t in range(T_max):
    mask = t_te == t
    if mask.sum() < 2:
      continue
    out[t] = r2_score(y_te[mask], pred_te[mask])
  return out


targets_for_curve = [("object_xy", "object_xy", None),
                     ("ee_xy",     "ee_xy",     None),
                     ("q_0",       "joints",    0)]
curves = {name: _r2_by_step(name, key, col, eps_tr, eps_te)
          for name, key, col in targets_for_curve}

fig, ax = plt.subplots(figsize=(9, 4))
for (name, _, _), c in zip(targets_for_curve, ("C0", "C1", "C2")):
  curve = curves[name]
  xs = np.arange(curve.shape[0])
  ax.plot(xs, curve, label=name, color=c, linewidth=2)
ax.axvline(burn_in - 0.5, color="k", linestyle=":", alpha=0.5, label="burn-in")
ax.set_xlabel("RSSM step index (open-loop starts after burn-in)")
ax.set_ylabel("R² (prior / hs / mlp probe)")
ax.set_title("Prior-rollout probe R² vs step")
ax.set_ylim(min(-0.05, ax.get_ylim()[0]), 1.05)
ax.axhline(0, color="k", linewidth=0.5, alpha=0.4)
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

## Summary table

Flat dump of every probe result. Useful for diffing across checkpoints —
if you re-run this notebook on different checkpoints and stash the JSON, the
schema is stable.

In [ ]:
def _aggregate(rows, target):
  return [{"source": r["source"], "latent": r["latent"], "probe": r["probe"],
           "r2": round(r["r2"], 4), "rmse": round(r["rmse"], 5)}
          for r in _by(rows, target=target)]

summary = {
  "checkpoint": str(checkpoint_path),
  "data":       str(data_path),
  "obs_mode":   obs_mode,
  "act_mode":   env.act_mode,
  "n_train_episodes": len(train_eps),
  "n_test_episodes":  len(test_eps),
  "burn_in":   burn_in,
  "horizon":   horizon,
  "object_xy": _aggregate(results, "object_xy"),
  "ee_xy":     _aggregate(results, "ee_xy"),
  "joints_mean_r2": {
    f"{src}/{key}/{probe}": float(np.mean([
      r["r2"] for r in results
      if r["source"] == src and r["latent"] == key and r["probe"] == probe
      and r["target"].startswith("q_")
    ])) for src in SOURCES for key in LATENT_KEYS for probe in PROBES
  },
}
print(json.dumps(summary, indent=2, default=str))